# Cut text-to-SQL prompt tokens by sending only the tables a question needs

A text-to-SQL prompt carries the database schema. Send all of it and every question pays for every
table, on every call. On a real warehouse that is tens of thousands of tokens, most of them about
tables the question never touches.

This notebook sends the model only the tables a question needs, picked by
[schemagate](https://github.com/ashishsinha1602/schemagate) (an open-source, offline selector:
BM25 plus a small embedder, no model call). It counts the difference with `tiktoken`, then has the
model write the SQL with the Responses API and runs it.

What you will see:

1. the full schema against the selected schema, in input tokens;
2. the model answering from the selection, with the SQL run against a real SQLite database;
3. the selection respecting who is asking: a table the caller may not read never enters the prompt;
4. what the saving is worth per month, and when it is not worth much.

Sections 1, 2, 4 and 5 run offline. Section 3 calls the API once and needs `OPENAI_API_KEY`
(a fraction of a cent).

In [ ]:
%pip install -q "schemagate>=1.15" openai tiktoken

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import sqlite3

import tiktoken
from openai import OpenAI
from schemagate import Principal
from schemagate.demo_schema import demo_catalog

MODEL = os.environ.get("OPENAI_MODEL", "gpt-5-mini")
enc = tiktoken.get_encoding("o200k_base")  # the tokenizer of the GPT-4o and GPT-5 families

## 1. A schema with more tables than any question needs

The bundled demo schema has 42 tables and views across a commerce, billing, HR and inventory
database, with the traps real schemas have: near-duplicate tables, cryptic legacy names, join tables
a question never mentions. It is materialised as a SQLite file with a few rows, so the SQL can run.

In [ ]:
cat = demo_catalog()
objects = list(cat.objects())
full_schema = "\n\n".join(o.render_ddl(max_columns=10**6) for o in objects)
print(len(objects), "objects,", len(full_schema), "characters of DDL")

42 objects, 8823 characters of DDL


## 2. Full schema against selection, in input tokens

The prompt is the same in both cases except for the schema block. `tiktoken` counts the prompt
text offline; section 3 shows the count the API reports for the request it actually sends.

In [ ]:
QUESTION = "Which customers owe us money, and how much?"


def prompt(schema: str, question: str) -> str:
    return (
        f"Database schema (SQLite):\n\n{schema}\n\nQuestion: {question}\n\n"
        "Reply with one SQLite SELECT and nothing else."
    )


def tokens(text: str) -> int:
    return len(enc.encode(text))


finance = Principal("okta:analyst", roles={"finance"})
selection = cat.select(QUESTION, top_k=6, principal=finance)
selected_schema = selection.prompt_fragment()

full = tokens(prompt(full_schema, QUESTION))
small = tokens(prompt(selected_schema, QUESTION))
print("tables sent:", [t.split(".")[-1] for t in selection.table_names])
print(f"full schema: {full:,} tokens")
print(f"selection:   {small:,} tokens  ({1 - small / full:.0%} fewer)")

tables sent: ['crm_customer', 'v_customer_balance', 'crm_contact', 'crm_opportunity', 'sales_order', 'sales_quote', 'core_party', 'core_address', 'sales_order_status']
full schema: 2,221 tokens
selection:   566 tokens  (75% fewer)


## 3. The model answers from the selection

The SQL runs against the demo database, so you can check the answer is real, not just plausible.
`usage.input_tokens` is what the request was billed for.

In [ ]:
client = OpenAI()  # reads OPENAI_API_KEY from the environment


def ask(schema: str, question: str):
    response = client.responses.create(model=MODEL, input=prompt(schema, question))
    sql = response.output_text.strip()
    sql = sql.removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
    return sql, response.usage.input_tokens


sql, billed = ask(selected_schema, QUESTION)
print(sql)
print("input tokens billed:", billed)

db = sqlite3.connect(cat._demo_url.removeprefix("sqlite:///"))
rows = db.execute(sql).fetchall()
print(len(rows), "rows, first five:")
for row in rows[:5]:
    print(tuple(round(v, 2) if isinstance(v, float) else v for v in row))

## 4. Selection is also access control

`hr_compensation` is readable only by the `payroll` role. For anyone else it is not ranked low; it is
absent, so the model cannot write a query against it or mention that it exists.

In [ ]:
SALARY = "salary and pay grade by employee"
for who in (Principal("okta:analyst", roles={"finance"}), Principal("okta:hr", roles={"payroll"})):
    tables = [t.split(".")[-1] for t in cat.select(SALARY, top_k=6, principal=who).table_names]
    print(sorted(who.roles), "->", "hr_compensation" in tables, tables)

['finance'] -> False ['hr_employee', 'v_employee_headcount', 'sales_order_status', 'v_monthly_revenue', 'billing_tax_rate', 'core_party', 'hr_department', 'core_country', 'core_address']
['payroll'] -> True ['hr_compensation', 'hr_employee', 'v_employee_headcount', 'sales_order_status', 'v_monthly_revenue', 'billing_tax_rate', 'core_party', 'hr_department', 'core_country']


## 5. What it is worth, and when it is not

Replace the volume and the price with your own. The selection stays at a handful of tables (the top
six, plus the tables they join to) however large the schema grows, so the saving grows with the
schema.

In [ ]:
questions_per_month = 1_000_000
price_per_million_input = 0.25  # USD, put your model's input price here


def monthly(tokens_per_question: int) -> float:
    return tokens_per_question * questions_per_month / 1e6 * price_per_million_input


print(f"full schema: ${monthly(full):,.0f}/month")
print(f"selection:   ${monthly(small):,.0f}/month")

full schema: $555/month
selection:   $142/month


On public benchmarks, measured the same way (no model called; the script and per-database numbers
are in schemagate's
[BENCHMARKS.md](https://github.com/ashishsinha1602/schemagate/blob/main/BENCHMARKS.md#prompt-tokens-on-public-schemas-the-saving-grows-with-the-schema)):

| Schemas | Median tables | Median saving per database |
|---|---|---|
| BIRD dev | 7 | 8.8% |
| Spider 2.0-lite BigQuery and Snowflake | 11 | 44.3% |
| the same, databases with 50+ tables | | 91.1% |

A seven-table database is already a few hundred tokens, so there is little to cut. Selection pays off
on big schemas. It also has a price: a table the question needs can be left out (on the Spider 2.0
warehouses, every gold table was selected for 86.7% of questions), so measure recall on your own
questions before relying on it.

## Where to go next

- `cat = Catalog().bootstrap("postgresql://...")` reflects your own database instead of the demo.
- `SchemagateRetriever` (`pip install langchain-schemagate`) does the same selection inside a
  LangChain or LangGraph agent.